In [2]:
import camelot
import pandas as pd
import numpy as np

**AXIS BANK**

In [ ]:
def _parse_axis_bank(self):
        """
        Parses Axis Bank statements using Camelot's lattice method.
        Standardizes columns and removes non-transaction rows.
        """
        # 1. Extract tables using lattice (grid-based)
        df = get_tables(self.file_path, flav='lattice')

        if df.empty:
            return pd.DataFrame() 

        # 2. Map the columns based on the Axis format in the images
        # Axis format: Tran Date | Chq No | Particulars | Debit | Credit | Balance | Init. Br
        df.columns = ["Date", "Chq No", "Particulars", "Debit", "Credit", "Balance", "Init_Br"]

        # 3. Clean up empty cells
        df.replace('', np.nan, inplace=True)

        # 4. Filter out Header Rows (removes rows where 'Date' column literally says 'Tran Date')
        df = df[~df['Date'].astype(str).str.contains('Tran Date|Date', case=False, na=False)]

        # 5. Filter out non-transaction rows (Opening/Closing balances and Totals)
        # We look at the 'Particulars' column for these keywords
        unwanted_keywords = 'TRANSACTION TOTAL|CLOSING BALANCE|OPENING BALANCE'
        df = df[~df['Particulars'].astype(str).str.contains(unwanted_keywords, case=False, na=False)]

        # 6. Drop rows that are completely empty (just a safety net)
        df.dropna(how='all', inplace=True)

        # 7. Reset index for a clean dataframe
        df.reset_index(drop=True, inplace=True)

        # We return only the standard columns required by the Dispatcher, dropping 'Init_Br'
        standard_cols = ["Date", "Particulars", "Chq No", "Debit", "Credit", "Balance"]
        return df[standard_cols]

In [ ]:
def _parse_bob(self):
        """
        Parses Bank of Baroda statements using the 'stream' method.
        Uses balance-differential logic to accurately sort Debits and Credits.
        """
        # 1. Extract tables using stream (text-based guessing)
        df_main = get_tables(self.file_path, flav="stream")

        if df_main.empty:
            return pd.DataFrame()

        # 2. Filter out the massive amount of header/footer junk BOB adds to every page
        unwanted_patterns = [
            'Transaction Details Page', '-----------------', 'Page Total', 
            'Grand Total:', 'Note: Cheques received', 'returning on the basis',
            'Unless the constituent', 'BANK OF BARODA', 'PALI ROAD', 
            'ADDRESS:', 'HELPLINE NO.', 'BRANCH PHONE NO.', 'MICR CODE:', 
            'A/C Number', 'Statement of account', 'DATE PARTICULARS',
            'We are committed', 'Please contact your branch', 'ABBREVIATIONS USED',
            'Pending penal charges', '\*\*\*\*END OF STATEMENT\*\*\*\*', 'As On',
            'A/C Name', 'City', 'CKYC Number', 'Tel No.', 'Nomination Flag',
            'Scheme Description', 'Joint Holders'
        ]
        
        regex_pattern = '|'.join(unwanted_patterns)
        
        # Keep only rows that DO NOT contain the unwanted patterns
        # We assume Camelot dumped the main text block into column 0
        cleaned_df = df_main[~df_main[0].astype(str).str.contains(regex_pattern, na=False, case=False)]
        cleaned_df.reset_index(drop=True, inplace=True)

        processed_rows = []
        previous_balance_numeric = None
        i = 0

        # 3. Iterate through the text lines to reconstruct the transactions
        while i < len(cleaned_df):
            # Clean up any leading numbers/spaces Camelot might have injected
            current_line = re.sub(r'^\d+\s+', '', str(cleaned_df[0].iloc[i]).strip())

            # Check if the line starts with a Date (DD-MM-YY format for BOB)
            if re.match(r'^\d{2}-\d{2}-\d{2}', current_line):
                full_transaction_line = current_line
                
                # --- Multi-line Description Merger ---
                # Check the next line to see if it's the continuation of a UPI description
                if (i + 1) < len(cleaned_df):
                    next_line = re.sub(r'^\d+\s+', '', str(cleaned_df[0].iloc[i + 1]).strip())
                    # If the next line DOES NOT start with a date, it belongs to the current transaction
                    if not re.match(r'^\d{2}-\d{2}-\d{2}', next_line) and next_line != "":
                        parts = current_line.split()
                        # Inject the next line into the middle of the current line's particulars
                        full_transaction_line = f"{' '.join(parts[:-2])} {next_line} {' '.join(parts[-2:])}"
                        i += 1 # Skip the next line in the main loop since we absorbed it

                # --- Parsing the reconstructed line ---
                parts = full_transaction_line.split()
                if len(parts) < 3:
                    i += 1
                    continue

                date = parts[0]
                balance_str = parts[-1]
                
                debit = np.nan
                credit = np.nan
                chq_no = ""
                
                # Handle the "Brought Forward" (B/F) starting row
                if "B/F" in parts:
                    description = 'Brought Forward'
                    previous_balance_numeric = clean_currency(balance_str)
                else:
                    amount_str = parts[-2]
                    # Everything between the Date and the Amount is the Description/Chq No
                    description = ' '.join(parts[1:-2])
                    
                    amount_numeric = clean_currency(amount_str)
                    current_balance_numeric = clean_currency(balance_str)

                    # --- Your Brilliant Balance-Differential Logic ---
                    if previous_balance_numeric is not None and current_balance_numeric is not None:
                        # If current balance is lower, money left the account (Debit)
                        if current_balance_numeric < previous_balance_numeric:
                            debit = amount_numeric
                        else:
                            credit = amount_numeric
                    
                    previous_balance_numeric = current_balance_numeric

                processed_rows.append({
                    'Date': date,
                    'Particulars': description,
                    'Chq No': chq_no, # BOB often merges this into particulars, so we leave blank to keep columns aligned
                    'Debit': debit,
                    'Credit': credit,
                    'Balance': balance_str
                })
            i += 1

        result_df = pd.DataFrame(processed_rows)
        
        # 4. Standardize output columns for the Dispatcher
        standard_cols = ["Date", "Particulars", "Chq No", "Debit", "Credit", "Balance"]
        for col in standard_cols:
            if col not in result_df.columns:
                result_df[col] = np.nan
                
        return result_df[standard_cols]